# M3: EV Fleet Projection 2027
## IE Sustainability Datathon March 2026, Iberdrola Challenge

**Objective:** Project the total EV fleet in Spain for 2027 (`total_ev_projected_2027`), a mandatory input for File 1.csv.

**Methodology:** Uses the SARIMA forecasting approach from the mandatory datos.gob.es fork:  
→ **Fork URL:** https://github.com/NOSIEMPRE/Laboratorio-de-Datos  
→ **Original notebook:** `Data Science/Ruta a la electrificación de la Movilidad/Codigo/Notebook.ipynb`

**This notebook:**
1. Replicates the SARIMA methodology from the fork (same model specification)
2. Extends the forecast from 12 months (2024) to 48 months (2024–2027)
3. Computes `total_ev_projected_2027` = cumulative historical fleet + projected registrations 2024–2027

**Scope:** Pure battery-electric vehicles (DGT propulsion code `'6'` / label `'Electrico'`), consistent with the fork's BEV definition.

---
**Data source:** DGT vehicle registrations via datos.gob.es fork (2015–2023 pre-processed parquets)

## 0. Setup: Install Dependencies

In [1]:
#!pip install statsmodels pandas numpy matplotlib pyarrow -q

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import curve_fit

warnings.filterwarnings('ignore')

FORK_URL = 'https://github.com/NOSIEMPRE/Laboratorio-de-Datos'
EV_CODE  = '6'   # COD_PROPULSION_ITV: 6 = Eléctrico (pure BEV)
print(f'Fork reference: {FORK_URL}')
print('Dependencies loaded.')

## 1. Download Historical Data from Fork (2015–2023)

Pre-processed parquet files (DGT registrations) are stored directly in the fork. This cell downloads any missing files using the same path structure as the original notebook.

In [3]:
PARQUET_DIR = "../Data/raw/ev_fleet_projections_datosgob/parquet"
os.makedirs(PARQUET_DIR, exist_ok=True)

ano_inicio, ano_fin = 2015, 2023

archivos_parquet = [
    (FORK_RAW_BASE + f"{ano}_{str(mes).zfill(2)}.parquet",
     f"{ano}_{str(mes).zfill(2)}.parquet")
    for ano in range(ano_inicio, ano_fin + 1)
    for mes in range(1, 13)
]

print(f"Files expected: {len(archivos_parquet)}")

downloaded, skipped, failed = 0, 0, []
for url, fname in archivos_parquet:
    fpath = os.path.join(PARQUET_DIR, fname)
    if os.path.exists(fpath):
        skipped += 1
        continue
    r = requests.get(url, timeout=30)
    if r.status_code == 200:
        with open(fpath, 'wb') as f:
            f.write(r.content)
        downloaded += 1
    else:
        failed.append(fname)
        print(f"  WARNING: {fname} returned HTTP {r.status_code}")

if failed:
    raise RuntimeError(f"Failed to download {len(failed)} file(s): {failed}")

print(f"Downloaded: {downloaded} | Already present: {skipped}")
n_parquet = len([f for f in os.listdir(PARQUET_DIR) if f.endswith('.parquet')])
print(f"Parquet files in dir: {n_parquet} (expected 108)")

Files expected: 108
Downloaded: 108 | Already present: 0
Parquet files in dir: 108 (expected 108)


## 2. Load & Filter Data

Filtering logic follows the fork exactly:
- Vehicle type: `COD_TIPO == '40'` (passenger cars; stored as string in parquet)
- Registration type: `CLAVE_TRAMITE` in `['1', '5', 'B']` (firm registrations)
- Propulsion mapping: DGT numeric code `'6'` → `'Electrico'` (pure BEV)
- EV filter: `COD_PROPULSION_ITV == 'Electrico'` (after numeric-to-label mapping)

In [4]:
dfs = []
for fname in sorted(os.listdir(PARQUET_DIR)):
    if fname.endswith('.parquet'):
        dfs.append(pd.read_parquet(os.path.join(PARQUET_DIR, fname)))

df_raw = pd.concat(dfs, ignore_index=True)
print(f"Total raw records loaded: {len(df_raw):,}")
print(f"Columns: {df_raw.columns.tolist()}")

Total raw records loaded: 14,588,498
Columns: ['FEC_MATRICULA', 'MARCA_ITV', 'MODELO_ITV', 'COD_TIPO', 'COD_PROPULSION_ITV', 'CLAVE_TRAMITE', 'CATEGORÍA_VEHÍCULO_ELÉCTRICO']


In [5]:
# DGT numeric propulsion codes → human-readable labels
cod_propulsion = {
    '0': 'Diesel',
    '1': 'Gasolina',
    '2': 'Hibrido Gasolina',
    '3': 'GNC',
    '4': 'GNL',
    '5': 'GLP',
    '6': 'Electrico',              # BEV, target category
    '7': 'Hidrogeno',
    '8': 'Gasolina-GNC',
    '9': 'Gasolina-GLP',
    'A': 'Hibrido Gasolina-Electrico',
    'B': 'Hibrido Diesel-Electrico',
    'C': 'Otro',
    'D': 'Diesel-GNC',
    'E': 'Diesel-GLP',
}

df = df_raw.copy()

df['COD_PROPULSION_ITV'] = df['COD_PROPULSION_ITV'].astype(str).str.strip()
df['COD_PROPULSION_ITV'] = df['COD_PROPULSION_ITV'].replace(cod_propulsion)

# Passenger cars only; COD_TIPO is stored as a string in these parquets
df['COD_TIPO'] = df['COD_TIPO'].astype(str).str.strip()
df = df[df['COD_TIPO'] == '40']
print(f"After COD_TIPO filter:      {len(df):>9,} records")

# Firm registrations only (excludes transfers and other non-sale events)
df['CLAVE_TRAMITE'] = df['CLAVE_TRAMITE'].astype(str).str.strip()
df = df[df['CLAVE_TRAMITE'].isin(['1', '5', 'B'])]
print(f"After CLAVE_TRAMITE filter: {len(df):>9,} records")

# Extract year and month from FEC_MATRICULA (format: DDMMYYYY)
df['ANO'] = df['FEC_MATRICULA'].astype(str).str[4:8].astype(int)
df['MES'] = df['FEC_MATRICULA'].astype(str).str[2:4].astype(int)

# Drop any records with an unrecognised propulsion type
df = df[df['COD_PROPULSION_ITV'].isin(cod_propulsion.values())]
print(f"After propulsion filter:    {len(df):>9,} records")
print(f"Year range: {df['ANO'].min()} – {df['ANO'].max()}")

print()
print('Propulsion breakdown (2023):')
print(df[df['ANO'] == 2023]['COD_PROPULSION_ITV'].value_counts())
print()
print(f"EV records total ('{EV_LABEL}'): "
      f"{(df['COD_PROPULSION_ITV'] == EV_LABEL).sum():,}")

After COD_TIPO filter:      10,379,959 records
After CLAVE_TRAMITE filter: 10,337,687 records
After propulsion filter:    10,332,822 records
Year range: 1923 – 2023

Propulsion breakdown (2023):
COD_PROPULSION_ITV
Diesel                        743834
Gasolina                      213284
Hibrido Gasolina               57381
Electrico                      27928
Hidrogeno                        295
Otro                              19
Gasolina-GNC                      14
Gasolina-GLP                      13
Hibrido Gasolina-Electrico         7
GNC                                6
Hibrido Diesel-Electrico           3
Name: count, dtype: int64

EV records total ('Electrico'): 113,624


## 3. Build Annual EV Series

Aggregate monthly registrations to annual totals — both total passenger cars and BEV. We use annual penetration rate (BEV / total) for the logistic fit rather than raw monthly counts, which avoids the seasonal noise problem that caused SARIMA to predict declining sales.

In [ ]:
df_bev   = df[df['COD_PROPULSION_ITV'] == EV_CODE].copy()

annual = pd.DataFrame({
    'total': df.groupby('ANO').size(),
    'bev':   df_bev.groupby('ANO').size(),
}).fillna(0).astype(int)
annual = annual[annual.index.between(2015, 2023)]
annual['penetration'] = annual['bev'] / annual['total']

print(f'Annual BEV registrations and market penetration:')
print(f'{"Year":>6}  {"Total cars":>12}  {"BEV":>8}  {"Penetration":>12}')
for yr, row in annual.iterrows():
    print(f'{yr:>6}  {row["total"]:>12,}  {row["bev"]:>8,}  {row["penetration"]:>11.2%}')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(annual.index, annual['bev'], color='steelblue')
ax.set_title('Annual BEV Registrations, Spain 2015–2023')
ax.set_ylabel('New BEV registrations')
plt.tight_layout(); plt.show()

## 4. Logistic Penetration Model

Fit a logistic S-curve to the annual BEV penetration rate:

$$p(t) = \frac{L}{1 + e^{-k(t - t_0)}}$$

- **L**: long-run maximum penetration (saturation level)
- **k**: growth rate steepness
- **t₀**: inflection year (fastest adoption)

Fit is applied on **2019–2023** data only, where policy support and charging infrastructure drove the structural acceleration. Pre-2019 data is noisy and reflects a different market regime.

In [ ]:
def logistic(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

fit_data  = annual[annual.index >= 2019]
years_fit = fit_data.index.astype(float).values
pens_fit  = fit_data['penetration'].values

popt, pcov = curve_fit(
    logistic, years_fit, pens_fit,
    p0=[0.30, 0.5, 2027],
    bounds=([0.10, 0.05, 2023], [1.0, 5.0, 2035]),
    maxfev=10000
)
L_fit, k_fit, t0_fit = popt
perr = np.sqrt(np.diag(pcov))

print('Logistic fit parameters:')
print(f'  L  (max penetration) = {L_fit:.4f}  ±{perr[0]:.4f}  ({L_fit*100:.1f}%)')
print(f'  k  (growth rate)     = {k_fit:.4f}  ±{perr[1]:.4f}')
print(f'  t0 (inflection year) = {t0_fit:.2f}  ±{perr[2]:.4f}')

t_smooth = np.linspace(2015, 2032, 300)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(fit_data.index, fit_data['penetration']*100, color='steelblue', zorder=5,
           s=60, label='Observed (fit range 2019–2023)')
ax.scatter(annual[annual.index < 2019].index, annual[annual.index < 2019]['penetration']*100,
           color='lightblue', zorder=4, s=40, label='Observed (pre-2019, not used in fit)')
ax.plot(t_smooth, logistic(t_smooth, *popt)*100, color='darkorange', lw=2, label='Logistic fit')
ax.axvline(2027, color='grey', ls=':', lw=1)
ax.scatter([2027], [logistic(2027,*popt)*100], color='red', s=80, zorder=6,
           label=f'2027 forecast: {logistic(2027,*popt)*100:.1f}%')
ax.set_xlabel('Year'); ax.set_ylabel('BEV market share (%)')
ax.set_title('BEV Market Penetration — Logistic Fit')
ax.legend(fontsize=8)

ax = axes[1]
proj_years = range(2024, 2028)
proj_pens_vals = {y: logistic(float(y), *popt) for y in proj_years}
all_pens = pd.concat([annual['penetration'], pd.Series(proj_pens_vals)])
bar_colors = ['steelblue']*len(annual) + ['darkorange']*len(proj_years)
ax.bar(all_pens.index, all_pens.values*100, color=bar_colors)
ax.set_xlabel('Year'); ax.set_ylabel('BEV market share (%)')
ax.set_title('Annual BEV Penetration Rate (observed + projected)')
ax.legend(handles=[
    mpatches.Patch(color='steelblue', label='Historical'),
    mpatches.Patch(color='darkorange', label='Projected')])

plt.tight_layout(); plt.show()

## 5. Extended Forecast: 2024–2027

Annual new BEV registrations = projected market size × logistic penetration rate.

Market size growth: 2% per year from 2023 base (conservative assumption aligned with ANFAC forecasts for total passenger car market recovery).

In [ ]:
base_market = int(annual['total'].iloc[-1])   # 2023 total car registrations
proj_bev = {
    y: int(base_market * (1.02 ** (y - 2023)) * logistic(float(y), *popt))
    for y in range(2024, 2028)
}

print('Projected annual BEV registrations:')
for y, n in proj_bev.items():
    print(f'  {y}: {n:>7,}  ({logistic(float(y),*popt)*100:.1f}% market share)')

fig, ax = plt.subplots(figsize=(12, 5))
all_bev = pd.concat([annual['bev'], pd.Series(proj_bev)])
n_hist  = len(annual)
colors  = ['steelblue']*n_hist + ['darkorange']*(len(all_bev)-n_hist)
ax.bar(all_bev.index, all_bev.values, color=colors)
ax.set_title('Annual BEV Registrations — Historical & Projected (Logistic Model)')
ax.set_ylabel('New BEV registrations')
ax.legend(handles=[
    mpatches.Patch(color='steelblue',  label='Historical (DGT parquets)'),
    mpatches.Patch(color='darkorange', label='Projected (logistic)')])
plt.tight_layout(); plt.show()

## 6. Total EV Fleet Projection: 2027

### Methodology
```
total_ev_projected_2027 = fleet_baseline_2023 + Σ projected_new_bev(2024..2027)
```
- `fleet_baseline_2023`: cumulative BEV registrations 2015–2023 (all still on road)
- `projected_new_bev`: logistic model × market size for each year 2024–2027

In [ ]:
fleet_baseline_2023     = int(annual['bev'].sum())
projected_2024_2027     = sum(proj_bev.values())
total_ev_projected_2027 = fleet_baseline_2023 + projected_2024_2027

print('=' * 50)
print('  EV FLEET PROJECTION — SPAIN 2027')
print('=' * 50)
print(f'  Cumulative BEV fleet (2015–2023):  {fleet_baseline_2023:>10,}')
print(f'  Projected registrations 2024–2027: {projected_2024_2027:>10,}')
print(f'  ─────────────────────────────────────────')
print(f'  TOTAL EV FLEET 2027:               {total_ev_projected_2027:>10,}')
print('=' * 50)
print(f'\n  → total_ev_projected_2027 = {total_ev_projected_2027}')

## 7. Year-by-Year Fleet Growth

In [ ]:
hist_annual = annual['bev']
proj_s      = pd.Series(proj_bev)
all_annual  = pd.concat([hist_annual, proj_s])
cumulative  = all_annual.cumsum()
n_hist      = len(hist_annual)
colors      = ['steelblue']*n_hist + ['darkorange']*(len(all_annual)-n_hist)

print('Annual registrations & cumulative fleet:')
summary = pd.DataFrame({
    'Annual': all_annual.astype(int),
    'Cumulative': cumulative.astype(int),
    'Type': ['Historical']*n_hist + ['Projected']*(len(all_annual)-n_hist)
})
print(summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(all_annual.index, all_annual.values, color=colors)
axes[0].set_title('Annual BEV Registrations, Spain')
axes[0].set_ylabel('New registrations')
axes[0].legend(handles=[
    mpatches.Patch(color='steelblue',  label='Historical'),
    mpatches.Patch(color='darkorange', label='Projected')])

axes[1].plot(cumulative.index, cumulative.values, marker='o', color='steelblue')
axes[1].axvline(x=2023.5, color='grey', ls=':', lw=1)
axes[1].scatter([2027], [total_ev_projected_2027], color='red', s=100, zorder=5,
                label=f'2027 Fleet: {total_ev_projected_2027:,}')
axes[1].set_title('Cumulative BEV Fleet, Spain (projected to 2027)')
axes[1].set_ylabel('Total vehicles')
axes[1].legend()

plt.tight_layout(); plt.show()

## 8. Output Verification

Verifies the final output value before it is used in **File 1.csv** (`total_ev_projected_2027`).

In [11]:
# ── Final output: value to use in File 1.csv ────────────────────────────────────────
print("OUTPUT VERIFICATION")
print("-" * 40)
print(f"Field:  total_ev_projected_2027")
print(f"Type:   {type(total_ev_projected_2027).__name__}")
print(f"Value:  {total_ev_projected_2027:,}")
print("-" * 40)
print(f"Source: SARIMA forecast extended from fork")
print(f"Fork:   {FORK_URL}")
print(f"Model:  SARIMAX(1,0,2)(1,0,1)[12] on log(monthly BEV registrations)")
print(f"Train:  Jan 2015 – Dec 2023 | Forecast: Jan 2024 – Dec 2027")
print("-" * 40)

assert isinstance(total_ev_projected_2027, int), \
    "ERROR: total_ev_projected_2027 must be integer"
assert total_ev_projected_2027 > 0, \
    "ERROR: value must be positive"
assert total_ev_projected_2027 > 100_000, \
    f"ERROR: value {total_ev_projected_2027:,} seems implausibly low for Spain 2027 BEV fleet"
print("✓ Type check passed (integer)")
print("✓ Value check passed (positive)")
print("✓ Sanity check passed (> 100,000)")

OUTPUT VERIFICATION
----------------------------------------
Field:  total_ev_projected_2027
Type:   int
Value:  220,171
----------------------------------------
Source: SARIMA forecast extended from fork
Fork:   https://github.com/Jvilpi/Laboratorio-de-Datos
Model:  SARIMAX(1,0,2)(1,0,1)[12] on log(monthly BEV registrations)
Train:  Jan 2015 – Dec 2023 | Forecast: Jan 2024 – Dec 2027
----------------------------------------
✓ Type check passed (integer)
✓ Value check passed (positive)
✓ Sanity check passed (> 100,000)


## 9. Export Output

Exports `total_ev_projected_2027` and model metadata to `../Data/processed/m3_ev_projection.json`. Downstream notebooks load this file when assembling **File 1.csv**.

In [12]:
import json as _json
from datetime import date

OUTPUT_PATH = '../Data/processed/m3_ev_projection.json'
os.makedirs('../Data/processed', exist_ok=True)

output = {
    'total_ev_projected_2027': total_ev_projected_2027,
    'fleet_baseline_2023': fleet_baseline_2023,
    'projected_registrations_2024_2027': projected_2024_2027,
    'logistic_params': {
        'L':  round(float(L_fit),  4),
        'k':  round(float(k_fit),  4),
        't0': round(float(t0_fit), 2),
    },
    'annual_forecast': {str(y): v for y, v in proj_bev.items()},
    'model': 'Logistic penetration curve fitted on 2019-2023',
    'generated_at': str(date.today()),
}

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    _json.dump(output, f, indent=2, ensure_ascii=False)

print(f'Saved: {OUTPUT_PATH}')
print()
print('Contents:')
print(_json.dumps(output, indent=2, ensure_ascii=False))

Saved: Data/processed/m3_ev_projection.json

Contents:
{
  "total_ev_projected_2027": 220171,
  "fleet_baseline_2023": 113624,
  "projected_registrations_2024_2027": 106547,
  "annual_forecast": {
    "2024-12-31 00:00:00": 32102,
    "2025-12-31 00:00:00": 28060,
    "2026-12-31 00:00:00": 24649,
    "2027-12-31 00:00:00": 21736
  },
  "model": "SARIMAX(1,0,2)(1,0,1)[12] on log(monthly BEV registrations)",
  "train_range": "Jan 2015 - Dec 2023",
  "fork": "https://github.com/Jvilpi/Laboratorio-de-Datos",
  "generated_at": "2026-04-11"
}
